```bash
conda activate
cd ~/link/other_model/learn_scGCN
file_name=scGCN_04_run_healthy
{
    jupyter-nbconvert scGCN__init__.ipynb ${file_name}.ipynb --to python
    conda activate scGCN
    # nohup python ${file_name}.py > nohup_${file_name} &
    # sleep 10 && rm ${file_name}.py
    conda activate
    echo 'finish'
}
```

In [1]:
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc

In [ ]:
from scGCN__init__ import scGCN,select_feature,scGCN_train
from scGCN__init__ import adata_norm_scale_for_scGCN,scGCN_get_res

In [2]:
map_sp = {k: v for k, v in zip(
    'h,m,z,ma,c,x'.split(','),
    'human,mouse,zebrafish,macaque,chicken,xenopus'.split(',')
)}
map_sp_reverse = {v: k for k, v in map_sp.items()}

map_sp.update({k: v for k, v in zip(
    'hs,mm'.split(','),
    'human,mouse'.split(',')
)})

p_root = Path('~/link/csMAHN_publish').expanduser()
p_res = p_root.joinpath("res")
p_cache = p_root.joinpath("cache")

# [from func.py]---------------------------------------------------------------------------
def get_path_varmap(
        sp_ref,
        sp_que,
        p_df_varmap=p_root.joinpath('homo/df_varmap.csv'),
        p_maps_SAMap=p_root.joinpath('homo/SAMap/maps_gene_name'),
        model='csMAHN'):
    """
    通过sp_ref 和 sp_que获取path_varmap
    ./homo/df_varmap.csv 存储了
    path_varmap路径及信息
"""
    p_df_varmap = Path(p_df_varmap)
    if model in 'scGCN'.split(','):
        df_varmap = pd.read_csv(p_df_varmap)
        index_ = df_varmap.query(
            "sp_ref == '{}' & sp_que == '{}'".format(
                sp_ref, sp_que)).index
        assert index_.size == 1, "[get {} path]can not get speicifed and unique path\nsp_ref\tsp_que\n{}\t{}".format(
            index_.size, sp_ref, sp_que)
        res = Path(df_varmap.loc[index_[0], 'path'])
        if not res.is_absolute():
            res = p_df_varmap.parent.joinpath(res)
        assert res.exists(), "[not exists] {}".format(res)
        return res

    elif model == 'SAMap':
        return p_maps_SAMap
    else:
        raise Exception(
            "[Error] can not find path_varmap with model '{}'".format(model))


def get_1v1_matches(
        df_match,
        key_homology_type='homology_type',
        value_homology_type='ortholog_one2one'):
    """
    from came.pp.take_1v_matches
    """
    l, r = df_match.columns[:2]
    l_unique = df_match[l].value_counts(
    ).to_frame('count').query("count == 1").index
    r_unique = df_match[r].value_counts(
    ).to_frame('count').query("count == 1").index
    keep = pd.DataFrame({
        'l_is_unique': df_match[l].isin(l_unique),
        'r_is_unique': df_match[r].isin(r_unique)
    }).min(axis=1)
    df_match = df_match[keep]
    df_match = df_match.query(
        "{} == '{}'".format(
            key_homology_type,
            value_homology_type))
    return df_match


def df_varmap_query_exists(
        df_varmap,
        list_gn_ref=[],
        list_gn_que=[],
        model='both'):
    df_varmap = df_varmap.copy()
    df_varmap['gn_ref_exists'] = df_varmap['gn_ref'].isin(list_gn_ref)
    df_varmap['gn_que_exists'] = df_varmap['gn_que'].isin(list_gn_que)
    if model == 'both':
        df_varmap = df_varmap.query("gn_ref_exists & gn_que_exists")
    elif model == 'ref':
        df_varmap = df_varmap.query("gn_ref_exists")
    elif model == 'que':
        df_varmap = df_varmap.query("gn_que_exists")
    else:
        raise Exception('[Error] model must be one of both, ref, que')
    df_varmap = df_varmap.drop(
        columns='gn_ref_exists,gn_que_exists'.split(','))
    return df_varmap

def get_type_counts_info(adatas, key_class, dsnames):
    type_counts_list = []
    for i in range(len(adatas)):
        type_counts_list.append(pd.value_counts(adatas[i].obs[key_class]))
    counts_info = pd.concat(type_counts_list, axis=1, keys=dsnames)
    return counts_info

def aligned_type(adatas, key_calss):
    adata1 = adatas[0].copy()
    adata2 = adatas[1].copy()
    counts_info = get_type_counts_info(
        adatas, key_calss, dsnames=["reference", "query"]
    )
    print("----raw----")
    print(counts_info)
    counts_info = counts_info.dropna(how="any")
    print("----new----")
    print(counts_info)

    com_type = counts_info.index.tolist()
    adata1 = adata1[adata1.obs[key_calss].isin(com_type)]
    adata2 = adata2[adata2.obs[key_calss].isin(com_type)]
    return adata1, adata2

In [3]:
def archive_gzip(p_source,decompress=False,out_dir=None,remove_source = True,show=True):
    """gzip 的压缩和解压缩
    """
    def archive_gzip_default(p_source,decompress,out_dir,remove_source,show):
        import gzip
        from shutil import copyfileobj
        
        func_open_source,func_open_target = open,gzip.open
        if decompress:
            func_open_source,func_open_target = func_open_target,func_open_source
            
        with func_open_source(p_source, 'rb') as f_source:  
            with func_open_target(p_target, 'wb') as f_target:  
                copyfileobj(f_source, f_target)

    def archive_gzip_Linux(p_source,decompress,out_dir,remove_source,show):
        import os
        os.system('gzip -{}c {} > {}'.format(
            'd' if decompress else '',p_source,p_target)
        )
    import platform
    
    handel_func = {
        'Linux':archive_gzip_Linux
    }.setdefault(platform.system(),archive_gzip_default)
    
    p_source = Path(p_source)
    assert p_source.exists(),'[not exitst] {}'.format(p_source)
    
    out_dir = p_source.parent if out_dir is None else Path(out_dir)
    if decompress:
        assert p_source.match('*.gz'),'[Error] p_source must end with .gz when decompress=True'
        p_target = out_dir.joinpath(p_source.name[:-3])
    else:
        if p_source.name.endswith('.gz'):
            print("[archive_gzip] has commpress {}".format(p_source)) if show else None
            return
        p_target = out_dir.joinpath('{}.gz'.format(p_source.name))


    handel_func(p_source,decompress,out_dir,remove_source,show)
    print('[archive_gzip][{}compress] {} -> {}'.format(
            'de' if decompress else '',p_source.name,p_target.name)) if show else None
    p_source.unlink() if remove_source else None

def scGCN_load_adata(p_dir, prefix=''):
    def _load_json(p):
        return json.loads(p.read_text())

    def load_h5ad_from_mtx(p_dir, prefix=''):
        p_dir = Path(p_dir)

        if p_dir.joinpath('{}matrix.mtx.gz'.format(prefix)).exists():
            archive_gzip(p_dir.joinpath('{}matrix.mtx.gz'.format(prefix)),
                         decompress=True, remove_source=False, show=False)

        assert p_dir.joinpath('{}matrix.mtx'.format(prefix)).exists(
        ), '[not exists] {}matrix.mtx or {}matrix.mtx.gz\n in {}'.format(prefix, p_dir)

        adata = sc.read_10x_mtx(p_dir, prefix=prefix)

        p_dir.joinpath(
            '{}matrix.mtx'.format(prefix)).unlink() if p_dir.joinpath(
            '{}matrix.mtx.gz'.format(prefix)).exists() else None

        # obs.csv
        if p_dir.joinpath('{}obs.csv'.format(prefix)).exists():
            adata.obs = pd.read_csv(
                p_dir.joinpath(
                    '{}obs.csv'.format(prefix)),
                index_col=0)
        else:
            print('[not exists]{}obs.csv\nin {}'.format(prefix, p_dir))
        return adata

    p_dir = Path(p_dir)
    adata = None
    if p_dir.match("*.h5ad"):
        adata = sc.read_h5ad(p_dir)
    elif p_dir.is_dir() and (
        p_dir.joinpath('{}matrix.mtx'.format(prefix)).exists() or p_dir\
            .joinpath('{}matrix.mtx.gz'.format(prefix)).exists()
    ):
        adata = load_h5ad_from_mtx(p_dir, prefix)
    else:
        raise Exception("[can not load adata] {}".format(p_dir))

    # [load] spatial info: adata.obsm and adata.uns['spatial']
    # adata.obsm
    if p_dir.joinpath('{}obsm'.format(prefix)).exists():
        for p_obsm in p_dir.joinpath('{}obsm'.format(prefix)).iterdir():
            if not p_obsm.match('*csv'):
                continue
            adata.obsm[p_obsm.stem] = pd.read_csv(p_obsm).to_numpy()
    # adata.uns['spatial']
    if p_dir.joinpath('{}uns/spatial'.format(prefix)).exists():
        adata.uns['spatial'] = {}
        for p_uns_spatial in p_dir.joinpath('{}uns/spatial'.format(prefix)
                                            ).iterdir():

            dict_spatial = {'images': {}}
            for img in p_uns_spatial.joinpath('images').iterdir():
                dict_spatial['images'][img.stem] = plt.imread(img)

            for p in p_uns_spatial.iterdir():
                if p.match('*.json'):
                    dict_spatial[p.stem] = _load_json(p)

            adata.uns['spatial'][p_uns_spatial.stem] = dict_spatial

    return adata

In [4]:
p_root_model = Path("~/link/other_model/learn_scGCN").expanduser()

In [5]:
# [run scGCN]---------------------------------------------------------------------------

# def precess_after_scGCN(
#     resdir, tissue_name, sp1, sp2, is_display=False, **kvargs
# ):

def run_scGCN(
    path_adata1,
    path_adata2,
    key_class1,
    key_class2,
    sp1,
    sp2,
    tissue_name,
    path_varmap,
    limite_func=lambda adata1,adata2: (adata1,adata2),
    aligned=False,
    resdir_tag=".",
    resdir=Path('.'), **kvargs
):
    """
    version:0.0.5
    kvargs:
        n_epochs: int
            default,200 见scGCN
    """

    # Parameter settings
    n_epochs = sum(kvargs.setdefault("n_epochs", [200]))

    # setting directory for results
    if len(resdir_tag) > 0:

        resdir_tag = "{}_{}-corss-{};{}".format(
            tissue_name, sp1, sp2, resdir_tag)
    else:
        resdir_tag = "{}_{}-corss-{}".format(tissue_name, sp1, sp2)

    resdir = resdir.joinpath(resdir_tag)

    # 终止 判断
    p_finish = resdir.joinpath("finish")
    if p_finish.exists():
        # precess_after_came(resdir,tissue_name,sp1, sp2)
        print(
            "[has finish]{} {}".format(
                time.strftime('%y%m%d-%H%M', time.localtime()),
                resdir.name)
        )
        return
    print(
        "[start]{} {}".format(
            time.strftime('%y%m%d-%H%M', time.localtime()),
            resdir.name

        ))
    # return

    figdir = resdir.joinpath("figs")
    sc.settings.figdir = figdir
    resdir.mkdir(parents=True, exist_ok=True)

    finish_content = ["[strat] {}".format(time.time())]

    # # setting
    dsnames = (
        '{}_{}'.format(
            tissue_name, sp1), '{}_{}'.format(
            tissue_name, sp2))
    dsn1, dsn2 = dsnames

    # load data
    adata_raw1 = scGCN_load_adata(path_adata1)
    adata_raw2 = scGCN_load_adata(path_adata2)
    sc.pp.calculate_qc_metrics(adata_raw1,qc_vars=[],percent_top=[],
                                       log1p=False,inplace=True)
    sc.pp.calculate_qc_metrics(adata_raw2,qc_vars=[],percent_top=[],
                                       log1p=False,inplace=True)
    sc.pp.filter_genes(adata_raw1, min_cells=10)
    sc.pp.filter_genes(adata_raw2, min_cells=10)

    
    df_homo = pd.read_csv(path_varmap,names='gn_ref,gn_que,type'.split(','),
                          skiprows=1).dropna(axis=0)
    df_homo = get_1v1_matches(df_homo, 'type')
    df_homo = df_varmap_query_exists(df_homo, adata_raw1.var.index,
                                     adata_raw2.var.index, model='both')
    print('[msg] get {} one2one item'.format(df_homo.shape[0]))
    
    # the var_names (genes) of the ref_adata and query_adata 
    # must be consistent and in the same order.
    # 即 one2one
    adata_raw1 = adata_raw1[:, df_homo['gn_ref']].copy()
    adata_raw2 = adata_raw2[:, df_homo['gn_que']].copy()
    adata_raw2.var.index = df_homo['gn_ref'].to_numpy()
    
    key_class = key_class1
    if key_class not in adata_raw2.obs.columns:
        adata_raw2.obs[key_class] = ''
    
    # limite 进一步对adata进行限制，默认不操作直接返回
    adata_raw1, adata_raw2 = limite_func(adata_raw1, adata_raw2)

    # group_counts_unalign.csv
    pd.concat(
        [
            adata_raw1.obs[key_class1].value_counts(),
            adata_raw2.obs[key_class2].value_counts(),
        ],
        axis=1,
        keys=dsnames,
    ).to_csv(resdir.joinpath("group_counts_unalign.csv"), index=True)
    # align
    if aligned:
        adata_raw1, adata_raw2 = aligned_type(
            [adata_raw1, adata_raw2], key_calss=key_class1
        )

    # 保存obs ,即真正测试的细胞的mata
    adata_raw1.obs.to_csv(resdir.joinpath("obs_ref.csv"), index=True)
    adata_raw2.obs.to_csv(resdir.joinpath("obs_que.csv"), index=True)


    # group_counts.csv
    temp = pd.concat(
        [
            adata_raw1.obs[key_class1].value_counts(),
            adata_raw2.obs[key_class2].value_counts(),
        ],
        axis=1,
        keys=dsnames,
    )
    temp.to_csv(resdir.joinpath("group_counts.csv"), index=True)
    if temp.shape[0] < 2:
        # 错误标记
        print("[Error][group_counts no any item]")
        finish_content.append(
            "[Error][group_counts no any item] %f" % time.time()
        )
        p_finish.with_name("error").write_text("\n".join(finish_content))
        return

    print("cell count --> {}".format(sum([adata_raw1.shape[0]+adata_raw2.shape[0]])))
    
    kvargs.update({'path_adata1': str(path_adata1),
                   'path_adata2': str(path_adata2),
                   'key_class1': key_class1,
                   'key_class2': key_class2,
                   'sp1': sp1,
                   'sp2': sp2,
                   'tissue_name': tissue_name,
                   'path_varmap': str(path_varmap),
                   'aligned': aligned,
                   'resdir_tag': resdir_tag,
                   'resdir': str(resdir),
                   'n_epochs':n_epochs}
                  )
    
    resdir.joinpath("kvargs.json").write_text(json.dumps(kvargs))


    finish_content.append("[finish before run] {}".format(time.time()))

    # run scGCN
    features = select_feature(adata_raw1, key_class)
    adata_raw1 = adata_raw1[:, features].copy()
    adata_raw2 = adata_raw2[:, features].copy()
    adata_raw1.write_h5ad(resdir.joinpath('adata_ref.h5ad'))
    adata_raw2.write_h5ad(resdir.joinpath('adata_que.h5ad'))
    
    # 将adata_1 adata_2 的副本标准化后传入graph_construct
    scGCN.graph.graph_construct(adata_norm_scale_for_scGCN(adata_raw1.copy()),
                                adata_norm_scale_for_scGCN(adata_raw2.copy()),
                                key_class, resdir)
    # 调用data.input_data
    scGCN.data.input_data(resdir, adata_raw1, adata_raw2,key_class)
    
    # epochs = 20 for test
    scGCN_train(resdir, resdir, epochs=n_epochs)
    finish_content.append("[finish run] {}".format(time.time()))
    
    # 后处理
    df_scGCN_res = scGCN_get_res(resdir,key_class,key_class,dsn1,dsn2)
    df_scGCN_res['UMAP1'] = 1
    df_scGCN_res['UMAP2'] = 1
    df_scGCN_res = df_scGCN_res.loc[:,
        'UMAP1,UMAP2,dataset,cell_type,true_label,pre_label,max_prob,is_right'.split(',')]
    df_scGCN_res.to_csv(resdir.joinpath('obs.csv'),index=True)
    # precess_after_scGCN(resdir, tissue_name, sp1, sp2,
    #                      is_display=False)
    finish_content.append("[finish after run] {}".format(time.time()))

    # 完成标记
    finish_content.append("[end] {}".format(time.time()))
    p_finish.write_text("\n".join(finish_content))

map_func_run_cross_species_models = {
    'scGCN': run_scGCN,

}

del run_scGCN
def run_cross_species_models(
    path_adata1,
    path_adata2,
    key_class1,
    key_class2,
    sp1,
    sp2,
    tissue_name,
    resdir,
    resdir_tag="",
    aligned=False,
    limite_func=lambda adata1, adata2: (adata1, adata2),
    models=''.split(','),
    **kvargs
):
    """
        models: scGCN
        kvargs:
        n_epochs:
            default, 200
        is_1v1: bool
            default,True scGCN仅one2one
    """
    
    for model in models:
        path_varmap = get_path_varmap(map_sp[sp1], map_sp[sp2], model=model)
        print('[path_varmap] {}\t{}'.format(model, Path(path_varmap).name))
        map_func_run_cross_species_models[model](
            path_adata1,
            path_adata2,
            key_class1,
            key_class2,
            sp1,
            sp2,
            tissue_name,
            path_varmap,
            aligned=aligned,
            resdir_tag=";".join([model, resdir_tag]),
            resdir=resdir,
            limite_func=limite_func,
            **kvargs,
        )

# run scGCN

In [6]:
q_item = 'HCL_MCA'.split(',')

df_para = pd.concat([pd.read_csv(p_cache.joinpath(
    'parameter_healthy_{}.csv'.format(i))).assign(mask=i)
    for i in q_item])
df_para['path_ref'] = df_para['path_ref'].apply(
    lambda x: p_cache.joinpath(x))
df_para['path_que'] = df_para['path_que'].apply(
    lambda x: p_cache.joinpath(x))

df_para

,tissue,sp_ref,path_ref,name_ref,sp_simple_ref,sp_que,path_que,name_que,sp_simple_que,key_cell_type,mask
0,Adrenal-Gland,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_adr,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_adr,m,CL,HCL_MCA
1,Bone-Marrow,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_bon,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_bon,m,CL,HCL_MCA
2,Brain,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_bra,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_bra,m,CL,HCL_MCA
3,Heart,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_hea,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_hea,m,CL,HCL_MCA
4,Intestine,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_int,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_int,m,CL,HCL_MCA
5,Kidney,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_kid,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_kid,m,CL,HCL_MCA
6,Liver,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_liv,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_liv,m,CL,HCL_MCA
7,Lung,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_lun,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_lun,m,CL,HCL_MCA
8,Spleen,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_spl,h,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_spl,m,CL,HCL_MCA
9,Adrenal-Gland,mouse,/public/workspace/licanchengup/link/csMAHN_pub...,m_adr,m,human,/public/workspace/licanchengup/link/csMAHN_pub...,h_adr,h,CL,HCL_MCA


> h_bon-map-m_bon 段错误

```txt
Segmentation fault (core dumped)
```

源于

```python

U, s, V = np.linalg.svd(mat)# 奇异值分解

```
段错误 ..... 啊这... 这只有在cpp里看见的, 这回居然再python里看见了

这 数组越界了？？？？




In [ ]:
# row = json.loads(Path('~/link/other_model').expanduser().joinpath("parameters_run_cross_species_models.json").read_text())
# row
n_epochs = [200]
for i,row in df_para.iloc[2:,:].iterrows():
    run_cross_species_models(
            path_adata1=row['path_ref'],
            path_adata2=row['path_que'],
            key_class1=row['key_cell_type'],
            key_class2=row['key_cell_type'],
            sp1=row['sp_simple_ref'],
            sp2=row['sp_simple_que'],
            tissue_name=row['tissue'],
            resdir_tag="{name_ref}-map-{name_que};is_1v1=True".format(**row),
            resdir=p_res,
            n_epochs=n_epochs,
            aligned=True,
            models = ['scGCN'])

print("\n[finish][run]\n".center(100,'-'))

In [7]:
print('\n[finish]\n'.center(100, '-'))

---------------------------------------------
[finish]
---------------------------------------------
